In [1]:
import sys
import os

print("Python utilisé par Jupyter:")
print(sys.executable)
print("\nVersion:")
print(sys.version)

# Remonter jusqu'au répertoire ATOMOD
notebook_dir = os.getcwd()
atomod_dir = os.path.abspath(os.path.join(notebook_dir, '..', '..'))

print(f"Répertoire notebook: {notebook_dir}")
print(f"Répertoire ATOMOD: {atomod_dir}")

sys.path.insert(0, atomod_dir)

# Maintenant l'import fonctionne
import HBPy
print("✅ HBPy importé!")




import logging
# Configuration du logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)
from pathlib import Path

from HBPy.Molecule.Crystal import Crystal,Atom

Python utilisé par Jupyter:
/home/bulou/src/essai/ATOMOD/venv/ATOMOD/bin/python3

Version:
3.10.12 (main, Mar  3 2026, 11:56:32) [GCC 11.4.0]
Répertoire notebook: /home/bulou/src/essai/ATOMOD/doc/tutorials
Répertoire ATOMOD: /home/bulou/src/essai/ATOMOD
✅ HBPy importé!


/home/bulou/src/essai/ATOMOD/venv/ATOMOD/lib/python3.10/site-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))


cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.


In [2]:
    status={
        'NP':True,
        'abtem':True,
        'feff':True,
        'atomic probability map':True,
        'optimization':False,
    }
    config={
        'run_dir':'run_dir',                          # répertoire de lancement
        'simul_dir':'simul',                          # répertoire de base de la simulation
        'train':{
            'TEM_img_dir'      : "train/TEM",        # répertoire de stockage des images TEM
            'EXAFS_dir'        : "train/EXAFS",      # répertoire de stockage des spectres EXAFS
            'prob_maps_img_dir': "train/prob_maps",  # répertoire de stockage des images TEM
            'nfo_dir'          : "train/nfo",        # répertoire des infos sur la gen. in silico
            'optimizer'        : "adam",
            'BATCH_SIZE'       :  4,
        },
        'NP':{
            'status':status['NP'],
            'seed':1,
            'structure':{
                'optimization':status['optimization'],
                'composition':['Pt','Co','Au','Pd','Rh'],
                'radius':5.0,
                'a':3.92,
            },
            'nvaccum':2.0,
        },
        'abtem':{
            'status':status['abtem'],
            'dx':0.04,
            'dy':0.04,
            'dz':4.08/2,
            'energy':300e3,
            'focal spread':40,
            'semiangle cutoff':20,
            'defocus':200,
            'cell scale':1.1
        },
        'atomic presence probability map':{
            'status':status['atomic probability map'],
            'ninter':{ # nombre d'intervalles entre deux positions atomiques
                'x':20,
                'y':20,
                'z':2
            },
            'sigma': .6  # en Å, largeur de la gaussienne ~ rayon atomique ou un peu moins
        },
        'image':{
            'xmin':0.0,
            'xmax':0.0,
            'ymin':0.0,
            'ymax':0.0,
            'H':128,
            'W':128,
            },
        'exafs':{
            'N_POINTS_EXAFS': 200,   # Nombre de points par spectre
        },
        'feff':{
            'status':status['feff'],
            'parameters':{
                'TITLE':'FEFF INPUT FILE',
                'DEBYE_TEMP': 190.0,
                'SCF_RADIUS': 5.0,
                'RPATH': 5.0,     # typique 2.2xdistance plus proches voisins. changer pour étudier la cvg des spectres
                'EXAFS' : 20.0,   # xkmax - default 20 ang.^-1
                'EDGE': {'Co':'K','Ni':'K','Ru':'K','Rh':'K','Pd':'K','Ir':'L3','Pt':'L3','Au':'L3'},
                'RMAX':8.0,
                'feff_dir':  '/home/bulou/ownCloud/Notebooks/M2P2_HEA/Home/Modelisation/ATOMOD/JFEFF/feff90/unix/',
                'input_save_dir':'./',
                'filename':'feff.inp',
                'list_pgm':['rdinp','atomic','dmdw','pot',
                            'opconsat', 
                            'screen',
                            'xsph',
                            'fms',
                            'mkgtr',
                            'path', 
                            'genfmt',
                            'ff2x',
                            'sfconv',
                            'compton',
                            'eels',
                            'ldos'
                            ]
            }
        }
    }
    

    config['run_dir']=Path.cwd()


_______________________________________
# Etape 1 : construire la nanoparticules
 _______________________________________
#   Etape 1.1 : la structure

In [3]:
NP=Crystal()
NP.build(a=config['NP']['structure']['a'],
    radius=config['NP']['structure']['radius'],
    materials='NP')
NP.origin_at_mass_center()
logger.info(f"min={NP.qmin} max={NP.qmax}")
logger.info(f"Mass center={NP.MC}")
logger.info(f"Number of atoms={len(NP.atoms)}")

2026-07-05 07:00:04,313 [INFO] - <module>() - min=[-2.94 -2.94 -2.94] max=[2.94 2.94 2.94]
2026-07-05 07:00:04,314 [INFO] - <module>() - Mass center=[ 4.44089210e-16 -3.48927236e-16 -2.53765263e-16]
2026-07-05 07:00:04,315 [INFO] - <module>() - Number of atoms=28


 #   Etape 1.2 : la distribution chimique   

In [4]:
NP.set_composition(config['NP']['structure']['composition'])
directory=config['run_dir']/config['simul_dir']/str(config['NP']['seed'])/config['train']['nfo_dir']/"XYZ"
NP.save(prefix="NP",fmt='xyz',savedir=directory)

#   Etape 1.3 : (optionnelle) l'optimisation structurale et/ou chimique

In [5]:
 if config['NP']['structure']['optimization']:
     NP.optimize_ase()
     logger.info(f"Optimization DONE!")

# Etape 1.4 : Visualiser la nanoparticule

In [6]:
import py3Dmol

chemin_fichier = directory/"NP.xyz"
xyz_data = chemin_fichier.read_text(encoding="utf-8")
vue = py3Dmol.view(width=400, height=400)
vue.addModel(xyz_data, "xyz")
vue.setStyle({'sphere': {'colorscheme': 'Jmol', 'scale': 1.0}, 'spacefill': {}})
vue.zoomTo()
vue.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.